In [1]:
#PAth setup for backend
import sys
import os

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', 'backend'))
sys.path.insert(0, ROOT)

print(f"Backend path: {ROOT}")

Backend path: c:\Users\parthsingh\CODES\intellicart_v2\backend


In [3]:
#importing necessary libraries
import torch
import torch.nn as nn
import numpy as np
import random
from database import SessionLocal
from models import Product, UserInteraction

In [5]:
#Load Data 
print("Loading data from Supabase...")

db = SessionLocal()

products     = db.query(Product).all()
interactions = db.query(UserInteraction).filter(
    UserInteraction.interaction_type == "purchase"
).all()

db.close()

print(f"Products: {len(products)}")
print(f"Interactions: {len(interactions)}")

Loading data from Supabase...
Products: 95
Interactions: 826


In [6]:
#Enoding product features
categories     = sorted(set(p.category for p in products))
category_to_idx = {cat: i for i, cat in enumerate(categories)}
num_categories  = len(categories)

def price_bucket(price):
    price = float(price)
    if price <= 50:    return 0
    elif price <= 100: return 1
    elif price <= 150: return 2
    elif price <= 200: return 3
    else:              return 4

num_price_buckets = 5
feature_dim       = num_categories + num_price_buckets

product_id_to_idx = {p.product_id: i for i, p in enumerate(products)}
idx_to_product    = {i: p for i, p in enumerate(products)}
num_products      = len(products)

def get_features(product):
    cat_onehot   = [0] * num_categories
    cat_onehot[category_to_idx[product.category]] = 1
    price_onehot = [0] * num_price_buckets
    price_onehot[price_bucket(product.price)] = 1
    return cat_onehot + price_onehot

feature_matrix = torch.tensor(
    [get_features(idx_to_product[i]) for i in range(num_products)],
    dtype=torch.float32
)

print(f"Feature matrix shape: {feature_matrix.shape}")
print(f"Categories: {categories}")

Feature matrix shape: torch.Size([95, 13])
Categories: ['Bakery', 'Beverages', 'Dairy', 'Fruits', 'Grains & Pulses', 'Personal Care', 'Snacks', 'Vegetables']


In [7]:
#Building training pairs
print("Building training pairs...")

user_purchases = {}
for interaction in interactions:
    uid = interaction.user_id
    pid = interaction.product_id
    if uid not in user_purchases:
        user_purchases[uid] = set()
    if pid in product_id_to_idx:
        user_purchases[uid].add(product_id_to_idx[pid])

positive_set   = set()
positive_pairs = []
for uid, indices in user_purchases.items():
    product_list = list(indices)
    for i in range(len(product_list)):
        for j in range(i + 1, len(product_list)):
            a, b = sorted([product_list[i], product_list[j]])
            if (a, b) not in positive_set:
                positive_set.add((a, b))
                positive_pairs.append((a, b, 1.0))

all_indices    = list(range(num_products))
negative_pairs = []
attempts       = 0
while len(negative_pairs) < len(positive_pairs) and attempts < 1000000:
    i = random.choice(all_indices)
    j = random.choice(all_indices)
    attempts += 1
    if i == j:
        continue
    a, b = sorted([i, j])
    if (a, b) not in positive_set:
        negative_pairs.append((i, j, 0.0))

all_pairs = positive_pairs + negative_pairs
random.shuffle(all_pairs)

print(f"Positive pairs: {len(positive_pairs)}")
print(f"Negative pairs: {len(negative_pairs)}")
print(f"Total pairs:    {len(all_pairs)}")

Building training pairs...
Positive pairs: 1838
Negative pairs: 1838
Total pairs:    3676


In [8]:
#Model Architecture
class ProductEmbeddingModel(nn.Module):
    def __init__(self, input_dim, embedding_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, embedding_dim)
        )

    def forward(self, x):
        return self.encoder(x)

    def get_embedding(self, x):
        with torch.no_grad():
            return self.encoder(x)

model = ProductEmbeddingModel(input_dim=feature_dim, embedding_dim=32)
print(model)

ProductEmbeddingModel(
  (encoder): Sequential(
    (0): Linear(in_features=13, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
  )
)


In [9]:
#Training Loop
print("Training...")

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn   = nn.BCEWithLogitsLoss()

EPOCHS     = 100
BATCH_SIZE = 128

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    random.shuffle(all_pairs)

    for batch_start in range(0, len(all_pairs), BATCH_SIZE):
        batch  = all_pairs[batch_start:batch_start + BATCH_SIZE]
        idx_a  = torch.tensor([p[0] for p in batch])
        idx_b  = torch.tensor([p[1] for p in batch])
        labels = torch.tensor([p[2] for p in batch], dtype=torch.float32)

        emb_a   = model(feature_matrix[idx_a])
        emb_b   = model(feature_matrix[idx_b])
        cos_sim = nn.functional.cosine_similarity(emb_a, emb_b) * 10
        loss    = loss_fn(cos_sim, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} — Loss: {total_loss:.4f}")

print("Training complete!")

Training...
Epoch 10/100 — Loss: 6.4504
Epoch 20/100 — Loss: 6.3175
Epoch 30/100 — Loss: 6.1601
Epoch 40/100 — Loss: 6.0974
Epoch 50/100 — Loss: 5.9240
Epoch 60/100 — Loss: 5.8114
Epoch 70/100 — Loss: 5.9523
Epoch 80/100 — Loss: 5.8962
Epoch 90/100 — Loss: 5.8446
Epoch 100/100 — Loss: 5.8321
Training complete!


In [10]:
#Saving the model and embeddings
model.eval()
all_embeddings = model.get_embedding(feature_matrix)

save_path = os.path.join(os.getcwd(), '..', 'models', 'model.pth')
save_path = os.path.abspath(save_path)

os.makedirs(os.path.dirname(save_path), exist_ok=True)

torch.save({
    "model_state":      model.state_dict(),
    "embeddings":       all_embeddings,
    "product_id_to_idx": product_id_to_idx,
    "idx_to_product":   {i: {"product_id": p.product_id, "name": p.name, "category": p.category, "price": float(p.price)} for i, p in idx_to_product.items()},
    "feature_matrix":   feature_matrix,
    "input_dim":        feature_dim,
    "embedding_dim":    32,
}, save_path)

print(f"Model saved to: {save_path}")

Model saved to: c:\Users\parthsingh\CODES\intellicart_v2\models\model.pth


In [11]:
#Sanity check: Find similar products to "Milk"
milk = next((p for p in products if p.name == "Milk"), None)
if milk:
    milk_idx = product_id_to_idx[milk.product_id]
    milk_emb = all_embeddings[milk_idx]

    cos_sims    = nn.functional.cosine_similarity(milk_emb.unsqueeze(0), all_embeddings)
    top_indices = cos_sims.argsort(descending=True).tolist()
    top_indices = [i for i in top_indices if i != milk_idx][:5]

    print("If you bought Milk, we recommend:")
    for idx in top_indices:
        p = idx_to_product[idx]
        print(f"  {p.name} ({p.category}) — similarity: {cos_sims[idx]:.4f}")

If you bought Milk, we recommend:
  Curd (Dairy) — similarity: 1.0000
  Buttermilk (Dairy) — similarity: 1.0000
  Cheese (Dairy) — similarity: 0.9840
  Ghee (Dairy) — similarity: 0.9840
  Condensed Milk (Dairy) — similarity: 0.9840
